# Parse House Roll Call PDF files

This notebook parses and saves data to CSV files from the House Roll Call vote PDFs that are stored in ```../../data/HJRollCalls'``` subfolders.

## Parsing a Single PDF File

First, I will parse the first House Roll Call Vote pdf in our dataset (stored at ```../../data/HJRollCalls/2003/hj01082003.pdf```)

This approach uses PyMuPDF to extract text from the PDF by line, then applies regular expressions to appropriately capture the data. This method works well because we know that the structure of the House Roll Call Vote Journals is consistent across the years, so regular expressions can capture the names of the different representatives as long as they are robust enough to handle some varied formatting.

### Setup

In [1]:
import pymupdf
import re
import pandas as pd
from pathlib import Path
import pytesseract
from pdf2image import convert_from_path
from datetime import datetime
import easyocr
from PIL import Image
from pdf2image import convert_from_path
from tempfile import TemporaryDirectory

Regular expressions needed to parse data are defined.

Depending on the year of the document, the required Regular expressions vary.

In [2]:
# Define regular expressions for validating a file's content
pre2009_patterns_to_match = [
    '^\s+MASSACHUSETTS HOUSE OF REPRESENTATIVES\s+$',
    '^\s+Yea and Nay\s+(?P<RCDate>\d{2}/\d{2}/\d{2})\s(?P<RCTime>\d{2}:\d{2})\s\D$',
    '^\s+No.\s+(?P<RCNumberForYear>\d+)\s+(?P<YeaVoteCount>\d+)\s+YEAS\s+(?P<NayVoteCount>\d+)\s+NAYS\s+(?P<NotVotingCount>\d+)\s+N/V\s+$',
    '^\s+(?P<Vote1>[Y,N,X,P])\s+(?P<Name1>Mr. Speaker|(.+?)(?:\s{,1}[A-Za-z.,\'-]+)+)\s+(?P<Vote2>[Y,N,X,P])\s+(?P<Name2>(.+?)(?:\s{,1}[A-Za-z.,\'-]+)+)\s+(?P<Vote3>[Y,N,X,P])\s+(?P<Name3>(.+?)(?:\s{,1}[A-Za-z.,\'-]+)+)\s+((?P<Vote4>[Y,N,X,P])\s+(?P<Name4>(.+?)(?:\s{,1}[A-Za-z.,\'-]+)+)\s+)?$',
    '^\s+\*=AFTER VOTE\s+P=PRESENT\s+X=NOT VOTING\s+$'
]

In [3]:
post2009_pre2017_patterns_to_match = [
    '^(‘)?\D+,\s(?P<VoteDate>\D+\d{1,2},\s\d{4}).*$',
    '^Yea and Nay No. (?P<RollCallCount>\d+)$'
]

In [4]:
post2017_patterns_to_match = [
    '^(?P<RCDate>\d{2}/\d{2}/\d{4})\s(?P<RCTime>\d{2}:\d{2} \D{2})$',
    '^\s+MASSACHUSETTS HOUSE OF REPRESENTATIVES$',
    '^(?P<VoteCount>\d+)\s(?P<VoteType>\D+)$',
    '^No.\s+(?P<RCNumberForYear>\d+)$',
    '^Yea and Nay$',
    '^(?P<Name>Mr. Speaker|([A-Z-][À-ÿa-z.,\'’-]+?)(?: {,1}[\*À-ÿA-Za-z.,\'’-]+)+)$',
    '^(?P<VoteValue>\D{1})$',
    '^\*=AFTER VOTE\s+P=PRESENT\s+X=NOT VOTING$'
]

There are a few files that specifically do not match the patterns seen in many of the others. It appears that although the document appears the same way compared to files from after 2017, the files from 2023 and onwards fix some of the encoding issues that resulted in Nay votes appearing out of order and the main title and other features of the document not appearing in expected order.

While a more robust solution can be expanded (or split into a separate function) for simplicity for now, the functionality is being controlled by tracking the years to except and will be updated in the future during clean-up phase of this project

In [5]:
years_to_except = [2023, 2024, 2025]

Parsing functions for the different years of PDFs are defined. Currently PDFs from before 2009 and after 2017 are supported and are defined in the following two functions.

In [6]:
def parse_Pre2009_PDF(path, bill_id, court_number, verbose=False):

    doc = pymupdf.open(path)
    all_pdf_bills = []
    all_pdf_votes = []

    for page in doc:
        text = page.get_text()
        lines = text.split('\n')

        current_pdf_bill_data = []
        current_pdf_bill_data.append(bill_id)
        current_pdf_bill_data.append(court_number)

        current_pdf_vote_data = []

        for i, line in enumerate(lines):
            if verbose: print(f'{i}: {line}')

            # Skip parsing if blank line
            if line.isspace():
                if verbose: print("Found blank line")
                continue

            if i == 2: # parse full title line individually for ease
                current_pdf_bill_data.append(line.strip())

            # match = re.search(title_regex, line)
            matched = False
            for j, pattern in enumerate(pre2009_patterns_to_match):
                match = re.match(pattern, line)

                if match:
                    matched = True
                    if verbose: print(f'Matched with pattern {pattern}')

                    if j == 0:
                        # Do nothing, just the title
                        if verbose: print("Parsing title line")
                    elif j == 1:
                        # Extract vote date and time
                        rc_date = match.group('RCDate')
                        current_pdf_bill_data.append(rc_date)
                        rc_time = match.group('RCTime')
                        current_pdf_bill_data.append(rc_time)
                        if verbose: print(f"Found date: {rc_date} and time: {rc_time}")
                    elif j == 2:
                        rc_count_for_year = match.group('RCNumberForYear')
                        current_pdf_bill_data.append(rc_count_for_year)

                        rc_yea_vote_count = match.group('YeaVoteCount')
                        current_pdf_bill_data.append(rc_yea_vote_count)

                        rc_nay_vote_count = match.group('NayVoteCount')
                        current_pdf_bill_data.append(rc_nay_vote_count)

                        rc_not_voting_count = match.group('NotVotingCount')
                        current_pdf_bill_data.append(rc_not_voting_count)

                        if verbose: print(f'Found {rc_yea_vote_count} YEA votes, {rc_nay_vote_count} Nay votes, {rc_not_voting_count} NV votes for RC Vote {rc_count_for_year}')
                    elif j == 3:
                        vote1 = match.group('Vote1')
                        name1 = match.group('Name1')
                        current_pdf_vote_data.append([bill_id, name1, vote1])

                        vote2 = match.group('Vote2')
                        name2 = match.group('Name2')
                        current_pdf_vote_data.append([bill_id, name2, vote2])

                        vote3 = match.group('Vote3')
                        name3 = match.group('Name3')
                        current_pdf_vote_data.append([bill_id, name3, vote3])

                        vote4 = match.group('Vote4')
                        name4 = match.group('Name4')
                        current_pdf_vote_data.append([bill_id, name4, vote4])

                        if verbose: print(f'Found {name1} with vote {vote1}, {name2} with vote {vote2}, {name3} with {vote3}')
                    elif j == 4:
                        if verbose: print("Parsing end line")

            if not matched:
                Exception(f"Failed to match regex on line {i} : {line}")

        all_pdf_bills.append(current_pdf_bill_data)
        all_pdf_votes.extend(current_pdf_vote_data)
        bill_id += 1

    new_bill_id = bill_id
    print(f"Succesfully parsed file at path: {path}")
    if verbose:
        print(f'Found {len(all_pdf_votes)} voting records')
        print(f'Found {len(all_pdf_bills)} bill records')
    return all_pdf_bills, all_pdf_votes, new_bill_id

In [7]:
def parse_2009through2017_PDF(path, bill_id, court_number, vote_counts, verbose=False):
    doc = pymupdf.open(path)
    all_pdf_bills = []

    for page_count, page in enumerate(doc):
        text = page.get_text()
        lines = text.split('\n')

        current_pdf_bill_data = []
        current_pdf_bill_data.append(bill_id)
        current_pdf_bill_data.append(court_number)
        current_pdf_bill_data.append('<TempName>')

        for i, line in enumerate(lines):
            if verbose: print(f'{i}: {line}')

            # Skip parsing if blank line
            if line.isspace():
                if verbose: print("Found blank line")
                continue

            # match = re.search(title_regex, line)
            rc_count_found = False
            rc_date_found = False

            for j, pattern in enumerate(post2009_pre2017_patterns_to_match):
                match = re.match(pattern, line)

                if match:
                    matched = True
                    if verbose: print(f'Matched with pattern {pattern}')

                    if j == 0:
                        # Extract vote date and convert to string
                        date_string = match.group('VoteDate')

                        # Define the format of your date string
                        date_format = "%B %d, %Y"

                        # Convert the string to a datetime object
                        datetime_object = datetime.strptime(date_string, date_format)

                        # Format the datetime object into a numerical string (e.g., YYYYMMDD)
                        numeric_date_string = datetime_object.strftime("%m/%d/%Y")

                        current_pdf_bill_data.append(numeric_date_string)

                        # Append a null value for the RC time for bills from this year
                        current_pdf_bill_data.append(None)

                        rc_date_found = True

                        if verbose: print(f"Found date: {numeric_date_string}")
                    elif j == 1:
                        # Extract roll call count
                        rc_count = match.group('RollCallCount')
                        current_pdf_bill_data.append(rc_count)
                        rc_date_found = True

                        if verbose: print(f"Found Roll Call Count: {rc_count}")


        if not rc_date_found or not rc_count_found:
            Exception(f"Failed to parse date or rc count in current page")

        # Append yay, nay, nv vote counts parsed from OCR method
        current_pdf_bill_data.append(vote_counts[page_count][0])
        current_pdf_bill_data.append(vote_counts[page_count][1])
        current_pdf_bill_data.append(vote_counts[page_count][2])

        all_pdf_bills.append(current_pdf_bill_data)
        bill_id += 1

    new_bill_id = bill_id
    print(f"Succesfully parsed file at path: {path}")
    if verbose:
        print(f'Found {len(all_pdf_bills)} bill records')
    return all_pdf_bills, new_bill_id

In [21]:
def parse_2009through2017_Votes_PDF(path, bill_id, verbose=False):

    pdf_pages = convert_from_path(path, 500)
    valid_votes = ['Y', 'N', 'X', 'P']

    all_pdf_vote_data = []
    all_bill_ids = []

    with TemporaryDirectory() as tempdir:
        pdf_pages = convert_from_path(path, 500)

        for page_enumeration, page in enumerate(pdf_pages, start=1):
            # enumerate() "counts" the pages for us.

            # Create a file name to store the image
            filename = f"{tempdir}\page_{page_enumeration:03}.jpg"

            # Save the image of the page in system
            page.save(filename, "JPEG")

            reader = easyocr.Reader(['en']) # Specify languages to recognize
            results = reader.readtext(Image.open(filename))

            all_bill_ids.append(bill_id)

            if verbose: print('Starting new page')

            for i, (bbox, text, prob) in enumerate(results):
                if verbose: print(f"Text: {text} Confidence: {prob}")
                current_vote = []
                if text in valid_votes:
                    current_vote.append(bill_id)

                    # Get vote value
                    vote = results[i][1]

                    # iterate until next vote found
                    j = i + 1
                    name = ''
                    while (j < len(results) and results[j][1] not in valid_votes):
                        name += results[j][1]
                        j += 1

                    current_vote.append(name)
                    current_vote.append(vote)
                    all_pdf_vote_data.append(current_vote)
            bill_id += 1

    vote_df = pd.DataFrame(all_pdf_vote_data, columns=['bill_id', 'voter_name', 'vote_value'])
    vote_dict = vote_df.groupby(['bill_id', 'vote_value']).count().rename(columns={'voter_name':'count'}).to_dict()

    all_vote_counts = []
    for id in all_bill_ids:
        curr_vote_counts = []
        yay_vote_count = 0 if vote_dict.get('count').get((id, 'Y')) is None else vote_dict.get('count').get((id, 'Y'))
        nay_vote_count = 0 if vote_dict.get('count').get((id, 'N')) is None else vote_dict.get('count').get((id, 'N'))
        nv_vote_count = 0 if vote_dict.get('count').get((id, 'X')) is None else vote_dict.get('count').get((id, 'X'))

        curr_vote_counts.append(yay_vote_count)
        curr_vote_counts.append(nay_vote_count)
        curr_vote_counts.append(nv_vote_count)

        all_vote_counts.append(curr_vote_counts)

    return all_pdf_vote_data, all_vote_counts

In [18]:
def parse_Post2017_PDF(path, bill_id, court_number, verbose=False):

    doc = pymupdf.open(path)
    all_pdf_bills = []
    all_pdf_votes = []

    file_year = int(str(path)[23:27])

    for page in doc:
        text = page.get_text()
        lines = text.split('\n')

        current_pdf_bill_data = []
        current_pdf_bill_data.append(bill_id)
        current_pdf_bill_data.append

        current_pdf_vote_data = []
        nay_vote_found = False
        end_found = False
        for i, line in enumerate(lines):
            if verbose: print(f'{i}: {line}')

            # Skip parsing if blank line
            if line.isspace():
                if verbose: print("Found blank line")
                continue

            if i == 7 and file_year not in years_to_except: # parse full title line individually for ease
                current_pdf_bill_data.insert(1, line.strip())
            elif file_year in years_to_except and i == 1:
                current_pdf_bill_data.insert(1, line.strip())

            # match = re.search(title_regex, line)
            matched = False
            if not end_found:
                for j, pattern in enumerate(post2017_patterns_to_match):
                    match = re.match(pattern, line)

                    if match:
                        matched = True
                        if verbose: print(f'Matched with pattern {pattern}')

                        if j == 0:
                            # Extract vote date and time
                            rc_date = match.group('RCDate')
                            current_pdf_bill_data.append(rc_date)
                            rc_time = match.group('RCTime')
                            current_pdf_bill_data.append(rc_time)
                            if verbose: print(f"Found date: {rc_date} and time: {rc_time}")
                        elif j == 1:
                            # Do nothing, just the title
                            if verbose: print("Parsing title line")
                            break
                        elif j == 2:
                            # Get vote count and type of vote since they appear each on a new line
                            rc_vote_count = match.group('VoteCount')
                            current_pdf_bill_data.append(rc_vote_count)

                            rc_vote_type = match.group('VoteType')

                            if verbose: print(f'Found {rc_vote_count} {rc_vote_type} votes')
                        elif j == 3:
                            # Parse RC Vote number
                            rc_count_for_year = match.group('RCNumberForYear')
                            current_pdf_bill_data.insert(3, rc_count_for_year) # Insert at index so it matches format of other PDF styles


                            if verbose: print(f'Found No. {rc_count_for_year} on current line')
                        elif j == 4:
                            # Parsing Yays and Nays text, do nothing
                            if verbose: print("Parsing \'Yays and Nays\'")
                            break
                        elif j == 5:
                            # Skip parsing this line if previous line was a Nay Vote
                            if nay_vote_found and file_year not in years_to_except:
                                nay_vote_found = False
                                print("Turning off nay vote found switch")
                                continue

                            # Parsing Voter Names
                            name = match.group('Name')

                            # Get Next line from list of lines to parse name for
                            vote_match = re.match('^(?P<VoteValue>\D{1})$', lines[i+1])
                            vote = vote_match.group('VoteValue')

                            current_pdf_vote_data.append([bill_id, name, vote])

                            if verbose: print(f'Found vote {vote} from Rep. {name}')
                        elif j == 6:
                            # Parse vote value, if value is N for Nay, parse name as well since they appear in reverse order

                            # Get Next line from list of lines to parse name for
                            vote_match = re.match('^(?P<VoteValue>\D{1})$', lines[i])
                            vote = vote_match.group('VoteValue')

                            if vote == 'N' and file_year not in years_to_except:
                                name_match = re.match('^(?P<Name>Mr. Speaker|([A-Z-][À-ÿa-z.,\'’-]+?)(?: {,1}[\*À-ÿA-Za-z.,\'’-]+)+)$', lines[i+1])
                                name = name_match.group('Name')

                                current_pdf_vote_data.append([bill_id, name, vote])
                                nay_vote_found = True
                                if verbose: print("Found Nay vote, will skip parsing next name")
                            else:
                                continue
                        elif j == 7:
                            if verbose: print("Parsing end line")
                            end_found = True
                            break
                if not matched:
                    Exception(f"Failed to match regex on line {i} : {line}")

        all_pdf_bills.append(current_pdf_bill_data)
        all_pdf_votes.extend(current_pdf_vote_data)
        bill_id += 1

    new_bill_id = bill_id
    print(f"Succesfully parsed file at path: {path}")
    if verbose:
        print(f'Found {len(all_pdf_votes)} voting records')
        print(f'Found {len(all_pdf_bills)} bill records')
    return all_pdf_bills, all_pdf_votes, new_bill_id

### Testing on a Single PDF document

In [27]:
bill_data = []
vote_data = []
bill_id = 0
path = "../../data/HJRollCalls/2013/hj04032013.pdf"

# Call defined function to get data for current pdf
pdf_vote_data, vote_counts_pdf = parse_2009through2017_Votes_PDF(path, bill_id)
pdf_bill_data, bill_id = parse_2009through2017_PDF(path, bill_id, 183, vote_counts_pdf)

bill_data.extend(pdf_bill_data)
vote_data.extend(pdf_vote_data)

/Users/gregoryknapp/Code/BU/DS701/ds-act-on-mass/venv/lib/python3.10/site-packages/torch/utils/data/dataloader.py:692: UserWarning: 'pin_memory' argument is set as true but not supported on MPS now, device pinned memory won't be used.
  warnings.warn(warn_msg)


Succesfully parsed file at path: ../../data/HJRollCalls/2013/hj04032013.pdf


In [28]:
vote_data

[[0, 'MrSpeaker', 'Y'],
 [0, 'Dykema', 'Y'],
 [0, 'Mark', 'Y'],
 [0, 'Toomey', 'Y'],
 [0, '#Mariano', 'Y'],
 [0, 'Ehrlich', 'Y'],
 [0, 'Markey', 'Y'],
 [0, 'Turner', 'X'],
 [0, 'Haddad', 'Y'],
 [0, 'Fallon', 'Y'],
 [0, 'McMur', 'Y'],
 [0, 'Vega', 'Y'],
 [0, 'Rushing', 'Y'],
 [0, 'Farley-Bouvier', 'Y'],
 [0, 'Miceli', 'Y'],
 [0, 'Wagner', 'Y'],
 [0, 'Reinstein', 'Y'],
 [0, 'Fennell', 'Y'],
 [0, 'Michlewitz', 'Y'],
 [0, 'WalshC', 'Y'],
 [0, 'Andrews', 'Y'],
 [0, 'Fernandes', 'Y'],
 [0, 'MoranF', 'Y'],
 [0, 'WalshM.', 'Y'],
 [0, 'Arciero', 'Y'],
 [0, 'Ferrante', 'Y'],
 [0, 'MoranM.', 'Y'],
 [0, 'WalshS', 'Y'],
 [0, 'Ashe', 'Y'],
 [0, 'Finn', 'Y'],
 [0, 'MurphyJ', 'Y'],
 [0, 'Zlotnik', 'Y'],
 [0, '*Atkins', 'Y'],
 [0, 'Forry', 'X'],
 [0, 'MurphyK', 'Y'],
 [0, 'Jones', 'Y'],
 [0, 'Ayers', 'Y'],
 [0, '#Fox', 'Y'],
 [0, 'Nangle', 'Y'],
 [0, 'Peterson', 'Y'],
 [0, 'Balser', 'Y'],
 [0, 'Fresolo', 'Y'],
 [0, 'Naughton', 'Y'],
 [0, 'Hill', 'Y'],
 [0, 'Basile', 'Y'],
 [0, 'Galvin', 'Y'],
 [0, 'Nym

In [29]:
bill_data

[[0, 183, '<TempName>', '04/03/2013', None, '41', 152, 0, 5],
 [1, 183, '<TempName>', '04/03/2013', None, '42', 29, 72, 4],
 [2, 183, '<TempName>', '04/03/2013', None, '43', 28, 78, 4],
 [3, 183, '<TempName>', '04/03/2013', None, '44', 118, 5, 8],
 [4, 183, '<TempName>', '04/03/2013', None, '45', 29, 72, 4],
 [5, 183, '<TempName>', '04/03/2013', None, '46', 116, 6, 6]]

In [30]:
# Create a dataframe from the parsed PDF data for bills
bill_df = pd.DataFrame(bill_data, columns=['bill_id', 'general_court_number', 'bill_name', 'rollcall_date', 'rollcall_time', 'rollcall_count_for_year', 'yea_vote_count', 'nay_vote_count', 'not_voting_count'])

bill_df.set_index('bill_id', inplace=True)
bill_df

,general_court_number,bill_name,rollcall_date,rollcall_time,rollcall_count_for_year,yea_vote_count,nay_vote_count,not_voting_count
bill_id,,,,,,,,
0,183,<TempName>,04/03/2013,None,41,152,0,5
1,183,<TempName>,04/03/2013,None,42,29,72,4
2,183,<TempName>,04/03/2013,None,43,28,78,4
3,183,<TempName>,04/03/2013,None,44,118,5,8
4,183,<TempName>,04/03/2013,None,45,29,72,4
5,183,<TempName>,04/03/2013,None,46,116,6,6


In [31]:
# Create a dataframe from parsed PDF data about votes
vote_df = pd.DataFrame(vote_data, columns=['bill_id', 'voter_name', 'vote_value'])
vote_df

,bill_id,voter_name,vote_value
0,0,MrSpeaker,Y
1,0,Dykema,Y
2,0,Mark,Y
3,0,Toomey,Y
4,0,#Mariano,Y
...,...,...,...
731,5,Malia,Y
732,5,Swan,Y
733,5,Dwyer,Y
734,5,Mannal,Y


## Batch Processing all PDFs

Now that we have a function for scraping the data from all pages of a single pdf, we can loop through all of the pdfs that are stored on disk in the data and create the full dataset

In [33]:
# Get path to all pdfs stored in HJRollCall folder
all_paths = []
root_path = Path('../../data/HJRollCalls')

for path in root_path.glob('**/*.pdf'):
    all_paths.append(path)

print(len(all_paths))

600


In [34]:
court_number_map = {
    2001: 182,
    2002: 182,
    2003: 183,
    2004: 183,
    2005: 184,
    2006: 184,
    2007: 185,
    2008: 185,
    2009: 186,
    2010: 186,
    2011: 187,
    2012: 187,
    2013: 188,
    2014: 188,
    2015: 189,
    2016: 189,
    2017: 190,
    2018: 190,
    2019: 191,
    2020: 191,
    2021: 192,
    2022: 192,
    2023: 193,
    2024: 193,
    2025: 194
}

### Parse all PDFs

This block is the core part of this notebook that actually iterates over the whole folder of PDFs and parses them.

Currently, this function is set to parse roll call votes that are either < 2009 or 2017 >. This is because PDFs from the years 2009 - 2017 are done is a specific style where the actual votes are stored as an image embedded in the PDF, not text, and require some special attention. In the worst case, this data may need to be manually scraped but future experiementation will be done to see if we can automate this process.

Each parse_####_PDF() function call takes 3 variables, path = the file path to parse, bill_id = the iterator of all bills parsed (used for reference in the database table), and an optional named parameter verbose, which is set to False by default. When verbose is set to True, the function will output debugging messages in case something goes wrong during the parsing process.

In [35]:
bill_id = -1
bill_data = []
vote_data = []

for path in all_paths:

    # If path includes the year <2009, skip for now
    file_year = int(str(path)[23:27])
    court_number = court_number_map[file_year]
    print(f'Attempting to parsing file at path: {path}')
    if file_year < 2009:
        current_bill_data, current_vote_data, bill_id = parse_Pre2009_PDF(path, bill_id+1, court_number)
    # For files from 2009 - 2017, parse the votes first since the data is needed for parsing the bill information
    elif file_year >= 2009 and file_year <= 2017:
        if bill_id == -1: bill_id = 0
        current_vote_data, current_vote_counts = parse_2009through2017_Votes_PDF(path, bill_id)
        current_bill_data, bill_id = parse_2009through2017_PDF(path, bill_id, court_number, current_vote_counts)
    elif file_year > 2017:
        current_bill_data, current_vote_data, bill_id = parse_Post2017_PDF(path, bill_id+1, court_number)
    else:
        print(f"File in unexpected year attempted to be parsed: {path, file_year}")
        break

    bill_data.extend(current_bill_data)
    vote_data.extend(current_vote_data)

Attempting to parsing file at path: ../../data/HJRollCalls/2013/hj04032013.pdf


/Users/gregoryknapp/Code/BU/DS701/ds-act-on-mass/venv/lib/python3.10/site-packages/torch/utils/data/dataloader.py:692: UserWarning: 'pin_memory' argument is set as true but not supported on MPS now, device pinned memory won't be used.
  warnings.warn(warn_msg)


Succesfully parsed file at path: ../../data/HJRollCalls/2013/hj04032013.pdf
Attempting to parsing file at path: ../../data/HJRollCalls/2013/hj07172013.pdf


/Users/gregoryknapp/Code/BU/DS701/ds-act-on-mass/venv/lib/python3.10/site-packages/torch/utils/data/dataloader.py:692: UserWarning: 'pin_memory' argument is set as true but not supported on MPS now, device pinned memory won't be used.
  warnings.warn(warn_msg)


Succesfully parsed file at path: ../../data/HJRollCalls/2013/hj07172013.pdf
Attempting to parsing file at path: ../../data/HJRollCalls/2013/hj09252013.pdf


/Users/gregoryknapp/Code/BU/DS701/ds-act-on-mass/venv/lib/python3.10/site-packages/torch/utils/data/dataloader.py:692: UserWarning: 'pin_memory' argument is set as true but not supported on MPS now, device pinned memory won't be used.
  warnings.warn(warn_msg)


... (265 outputs left out)

/Users/gregoryknapp/Code/BU/DS701/ds-act-on-mass/venv/lib/python3.10/site-packages/torch/utils/data/dataloader.py:692: UserWarning: 'pin_memory' argument is set as true but not supported on MPS now, device pinned memory won't be used.
  warnings.warn(warn_msg)


Succesfully parsed file at path: ../../data/HJRollCalls/2017/hj09272017.pdf
Attempting to parsing file at path: ../../data/HJRollCalls/2017/hj10112017.pdf


/Users/gregoryknapp/Code/BU/DS701/ds-act-on-mass/venv/lib/python3.10/site-packages/torch/utils/data/dataloader.py:692: UserWarning: 'pin_memory' argument is set as true but not supported on MPS now, device pinned memory won't be used.
  warnings.warn(warn_msg)


ValueError: time data 'Ocotober 11, 2017' does not match format '%B %d, %Y'

### Creating and writing DataFrames from parsed data then converting to CSV

In [60]:
bill_data[0]

[0, 'QUORUM', '06/08/2022', '01:36 PM', '194', '155', '0', '0']

In [ ]:
# Create a dataframe from the parsed PDF data for bills
bill_df = pd.DataFrame(bill_data, columns=['bill_id', 'general_court_number', 'bill_name', 'rollcall_date', 'rollcall_time', 'rollcall_count_for_year', 'yea_vote_count', 'nay_vote_count', 'not_voting_count'])

bill_df.set_index('bill_id', inplace=True)
bill_df

,bill_name,rollcall_date,rollcall_time,rollcall_count_for_year,yea_vote_count,nay_vote_count,not_voting_count
bill_id,,,,,,,
0,QUORUM,06/08/2022,01:36 PM,194,155,0,0
1,H. 4805 shall this pass notwithstanding Objection,06/08/2022,01:50 PM,195,119,36,0
3,QUORUM,07/13/2022,01:32 PM,212,154,0,0
4,"H. 5007 Consolidated Amendment ""A""",07/13/2022,05:49 PM,213,154,0,0
6,QUORUM,02/16/2022,01:50 PM,144,156,0,0
...,...,...,...,...,...,...,...
3567,H. 3899 On Enactment Land Taking,09/30/03,02:14,416,147,0,13
3568,QUORUM,09/30/03,02:55,417,150,0,10
3569,S. 187 On Engrossment,09/30/03,03:10,418,146,4,10


In [62]:
# Create a dataframe from parsed PDF data about votes
vote_df = pd.DataFrame(vote_data, columns=['bill_id', 'voter_name', 'vote_value'])
vote_df

,bill_id,voter_name,vote_value
0,0,Mr. Speaker,Y
1,0,Farley-Bouvier,Y
2,0,Mahoney,Y
3,0,Turco,Y
4,0,Hogan,Y
...,...,...,...
508949,3571,Webster,N
508950,3571,Festa,Y
508951,3571,Miceli,Y
508952,3571,"Sullivan, D.B.",Y


The parsed output can be written to csv files for future cleaning and processing

In [66]:
bill_df.to_csv('../../data/parsedcsv/hjbills_raw.tsv', index=True, index_label=bill_df.index.name, sep='\t')
vote_df.to_csv('../../data/parsedcsv/hjrollcallvotes_raw.tsv', index=False, sep='\t')

Read the files from CSV to test that they wrote properly

In [68]:
bill_df_from_csv = pd.read_csv('../../data/parsedcsv/hjbills_raw.tsv', sep='\t')

bill_df_from_csv.head()

,bill_id,bill_name,rollcall_date,rollcall_time,rollcall_count_for_year,yea_vote_count,nay_vote_count,not_voting_count
0,0,QUORUM,06/08/2022,01:36 PM,194,155.0,0.0,0.0
1,1,H. 4805 shall this pass notwithstanding Objection,06/08/2022,01:50 PM,195,119.0,36.0,0.0
2,3,QUORUM,07/13/2022,01:32 PM,212,154.0,0.0,0.0
3,4,"H. 5007 Consolidated Amendment ""A""",07/13/2022,05:49 PM,213,154.0,0.0,0.0
4,6,QUORUM,02/16/2022,01:50 PM,144,156.0,0.0,0.0


In [70]:
vote_df_from_csv = pd.read_csv('../../data/parsedcsv/hjrollcallvotes_raw.tsv', sep='\t')
vote_df_from_csv.head()

,bill_id,voter_name,vote_value
0,0,Mr. Speaker,Y
1,0,Farley-Bouvier,Y
2,0,Mahoney,Y
3,0,Turco,Y
4,0,Hogan,Y


## Debugging individual PDFs

The following section is used for debugging specific documents that did not work. It is functionally identical to the section ### Testing on a single document, but is kept down here for reference for the work process, and less for presentation purposes.

In [ ]:
path = '../../data/HJRollCalls/2013/hj04032013.pdf'
doc = pymupdf.open(path)

for page in doc:

    pix = page.get_pixmap()
    pix.pdfocr_save('../../data/HJRollCalls/2017/hj01252017_ocr.pdf')
    imgpdf = pymupdf.open("pdf", pix.pdfocr_tobytes())

    print("NEW DOC STARTING \n\n")

    for page in imgpdf:

        text = page.get_textpage_ocr()
        print(text.extractText())


NEW DOC STARTING 


Yea and Nay Supplement
JOURNAL OF THE HOUSE.
Wednesday, Match 22, 2017,
Yea and Nay No. 22
(On passing tobe engrossed the House Bill making appropsatios for
the fla
year 2017 10 provide for supplementing ceria existing
Sporopriatins apd for cerain ether actin and projects (House,
No. S448, amended.
Yeas: 186
Nowt
Nara
Yea and Nay Supplement
JOURNAL OF THE HOUSE.
Wednesday, Match 22, 2017,
Yea and Nay No. 22
(On passing tobe engrossed the House Bill making appropsatios for
the fla
year 2017 10 provide for supplementing ceria existing
Sporopriatins apd for cerain ether actin and projects (House,
No. S448, amended.
Yeas: 186
Nowt
Nara



Tested libraries for OCR Parsing:

* PaddleOCR
* easyocr
* pytesseract

### Pytesseract failed attempt to parse

In [15]:
path = Path('../../data/HJRollCalls/2013/hj04032013.pdf')

image_file_list = []



with TemporaryDirectory() as tempdir:
    pdf_pages = convert_from_path(path, 500)

    for page_enumeration, page in enumerate(pdf_pages, start=1):
        # enumerate() "counts" the pages for us.

        # Create a file name to store the image
        filename = f"{tempdir}\page_{page_enumeration:03}.jpg"

        # Declaring filename for each page of PDF as JPG
        # For each page, filename will be:
        # PDF page 1 -> page_001.jpg
        # PDF page 2 -> page_002.jpg
        # PDF page 3 -> page_003.jpg
        # ....
        # PDF page n -> page_00n.jpg

        # Save the image of the page in system
        page.save(filename, "JPEG")
        image_file_list.append(filename)

    for image_file in image_file_list:
        # Recognize the text as string in image using pytesserct
        text = str(((pytesseract.image_to_string(Image.open(image_file)))))

        # The recognized text is stored in variable text
        # Any string processing may be applied on text
        # Here, basic formatting has been done:
        # In many PDFs, at line ending, if a word can't
        # be written fully, a 'hyphen' is added.
        # The rest of the word is written in the next line
        # Eg: This is a sample text this word here GeeksF-
        # orGeeks is half on first line, remaining on next.
        # To remove this, we replace every '-\n' to ''.
        text = text.replace("-\n", "")

        print(text)

        break


HARK KK RRR KR KKK KR RRR KKK KR RRR RK RR RRR OK OK

KKK KK

Yea and Nay Supplement
JOURNAL OF THE HOUSE.

Wednesday, April 3, 2013.

Yea and Nay No. 41

On passing to be enacted the engrossed [land taking] Bill authorizing
the city of Springfield to convey certain park land and to grant a license
agreement over certain park land to Unifirst Corporation (see House,
No. 58).

Mr. Speaker Y Dykema Y Mark a Toomey
*Mariano ¥ Ehrlich Y Markey Turner
Haddad Y Fallon Y McMurtry Y Vega
Rushing Y Farley-Bouvier Y Miceli Y Wagner
Reinstein ¥ Fennell ¥ Michlewitz ¥ Walsh, C.
Andrews ¥ Fernandes ¥ Moran, F. ¥ Walsh, M.
Arciero ¥ Ferrante ¥ Moran, M. ¥ Walsh, S.
Ashe ¥ Finn ¥ Murphy, J. ¥ Zlotnik
*Atkins x Forry ¥ Murphy, K. Y¥ Jones
Ayers Y *Fox Y Nangle ¥ Peterson
Balser Y Fresolo Y Naughton Y Hill
Basile ¥ Galvin Y Nyman Y Poirier
Benson ¥ Garballey Y O' Day Y Barrows
Binienda Y Garlick Y O'Flaherty YX Beaton
Bradley Y Garry Y Parisella Y Boldyga
Brady ¥ Gobi ¥ Peake ¥ deMacedo
Brodeur ¥ Golde

### Easy OCR - Good library with decent enough accuracy to start but needs manual cleaning

In [84]:
import easyocr
from PIL import Image
from pdf2image import convert_from_path
from tempfile import TemporaryDirectory

path = '../../data/HJRollCalls/2017/hj02022017.pdf'

pdf_pages = convert_from_path(path, 500)
valid_votes = ['Y', 'N', 'X', 'P']

with TemporaryDirectory() as tempdir:
    pdf_pages = convert_from_path(path, 500)

    for page_enumeration, page in enumerate(pdf_pages, start=1):
        # enumerate() "counts" the pages for us.

        # Create a file name to store the image
        filename = f"{tempdir}\page_{page_enumeration:03}.jpg"

        # Save the image of the page in system
        page.save(filename, "JPEG")

        reader = easyocr.Reader(['en']) # Specify languages to recognize
        results = reader.readtext(Image.open(filename))

        print('Starting new page')
        found_votes = False
        vote_data = []
        bill_id = 1 # TODO: Parse actual bill number
        for i, (bbox, text, prob) in enumerate(results):
            print(f"Text: {text} Confidence: {prob}")
            current_vote = []
            if text in valid_votes:
                # Set flag if first time here
                found_votes = True

                # Get vote value
                vote = results[i][1]
                current_vote.append(vote)

                # iterate until next vote found
                j = i + 1
                name = ''
                while (j < len(results) and results[j][1] not in valid_votes):
                    name += results[j][1]
                    j += 1

                current_vote.append(name)
                vote_data.append(current_vote)
        break

    print(vote_data)


/Users/gregoryknapp/Code/BU/DS701/ds-act-on-mass/venv/lib/python3.10/site-packages/torch/utils/data/dataloader.py:692: UserWarning: 'pin_memory' argument is set as true but not supported on MPS now, device pinned memory won't be used.
  warnings.warn(warn_msg)


Starting new page
Text: Yea and Confidence: 0.9110323592148304
Text: Supplement Confidence: 0.9999409650353723
Text: JOURNAL OF THE HOUSE. Confidence: 0.9409396735929638
Text: Thursday, February 2, 2017. Confidence: 0.9026027254969788
Text: Yea and Confidence: 0.9184938287411333
Text: No. 6 Confidence: 0.9632047703532381
Text: On passing, notwithstanding the objections of His Excellency the Confidence: 0.8943664349858784
Text: Governor; the engrossed Bill  further regulating the compensation of Confidence: 0.6489101369998898
Text: certain public officials (see House, No. 58, amended) Confidence: 0.8372583329792943
Text: Y Confidence: 0.5914404727111702
Text: Mr Confidence: 0.9949265997531603
Text: Speaker Confidence: 0.9998234170676896
Text: Y Confidence: 0.3168961437906468
Text: Finn Confidence: 0.9997276067733765
Text: Y Confidence: 0.3307480366232909
Text: McMurtry Confidence: 0.9775741581388201
Text: Y Confidence: 0.3088046806333864
Text: Wagner Confidence: 0.8227890805676277
Text:

In [9]:
bill_data, vote_data, bill_id = parse_Post2017_PDF(path, 0, verbose=True)

NameError: name 'parse_Post2017_PDF' is not defined

In [57]:
# Create a dataframe from the parsed PDF data for bills
bill_df = pd.DataFrame(bill_data, columns=['bill_id', 'bill_name', 'rollcall_date', 'rollcall_time', 'rollcall_count_for_year', 'yea_vote_count', 'nay_vote_count', 'not_voting_count'])

bill_df.set_index('bill_id', inplace=True)
bill_df

,bill_name,rollcall_date,rollcall_time,rollcall_count_for_year,yea_vote_count,nay_vote_count,not_voting_count
bill_id,,,,,,,
0,H. 3854 notwithstanding the objection of the Gov,09/18/2019,02:17 PM,91,156,1,3
1,H. 4000 item 1595-6368 Transportation Trust Fund,09/18/2019,02:40 PM,92,125,31,4
2,H. 4000 item 2810-0100 Parks and Recreation,09/18/2019,02:45 PM,93,130,26,4
3,H. 4000 item 8900-0001 Department of Correction,09/18/2019,02:48 PM,94,125,31,4
4,H. 4076 On Engrossment,09/18/2019,03:13 PM,95,156,0,4


In [58]:
# Create a dataframe from parsed PDF data about votes
vote_df = pd.DataFrame(vote_data, columns=['bill_id', 'voter_name', 'vote_value'])
vote_df

,bill_id,voter_name,vote_value
0,0,Mr. Speaker,Y
1,0,Farley-Bouvier,Y
2,0,McGonagle,Y
3,0,Vega,Y
4,0,Mariano,Y
...,...,...,...
795,4,Wong,Y
796,4,Elugardo,Y
797,4,Markey,Y
798,4,Vargas,Y
